# 12 · Auto mode with both fine-tuned models

**SimClinic, CS425 Group 7.** In notebook 05 the doctors interview a rule-based card patient (`simclinic/patients.py`), because the patient model was not ready yet. This notebook runs the full system as the proposal describes it: **the fine-tuned GPT-2 doctor interviews the fine-tuned GPT-2 patient** on the 201 held-out test cards. The hidden disease is known, so every run scores both models at once:

- **doctor:** top-1 / top-3 accuracy, questions asked, and naive Bayes on the same revealed evidence (asking vs reasoning, as in notebook 05)
- **patient:** symptom fidelity, card coverage and disease leakage, now under a doctor that asks free-form questions instead of notebook 08's fixed script

Every doctor is also run against the rule-based patient on the same cards and seeds, so the two columns show how much the patient changes each doctor's score.

**Needs** the patient checkpoints from notebook 08 (`models/patient/<run>/` (notebook 08's output folder), trained on the SCIS cluster; not in the repository).

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

MAX_QUESTIONS = 10
SEED = 42
PATIENT_RUNS = {"GPT-2 small patient": "stage2-gpt2-lr0.0001", "GPT-2 medium patient": "stage2-gpt2-medium-lr5e-05"}   # runs that are missing are skipped
PATIENT_CKPT = ROOT / "models" / "patient"          # notebook 08's MODELS_DIR
DATA_DIR = ROOT / "data" / "processed" / "doctor"
CKPT_DIR = ROOT / "checkpoints" / "doctor"
OUT_DIR = DATA_DIR / "eval" / "auto_gpt2_patient"
RESULTS = ROOT / "results" / "auto_mode"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Setup

In [2]:
import gc
import json
import re
import time

import numpy as np
import pandas as pd
import torch
from transformers import set_seed

from simclinic import cards as C
from simclinic.consult import run_consultation, score
from simclinic.doctors import GPT2Doctor, NBDoctor
from simclinic.lexicon import Lexicon, revealed_symptoms
from simclinic.naive_bayes import SymcatNaiveBayes
from simclinic.patients import CardPatient
from simclinic.symcat import Condition

def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

CONDITIONS = json.loads((DATA_DIR / "conditions.json").read_text())
symcat_table = {n: Condition(**c) for n, c in json.loads((DATA_DIR / "symcat_conditions.json").read_text()).items()}
lexicon = Lexicon.load(symptoms=sorted({s for n in CONDITIONS for s in symcat_table[n].symptoms}))
nb = SymcatNaiveBayes({n: symcat_table[n].symptoms for n in CONDITIONS})
test_cards = read_jsonl(DATA_DIR / "cards" / "test.jsonl")
seed_of = {c["card_id"]: SEED + i for i, c in enumerate(test_cards)}       # notebook 05's seeds
grid = pd.read_csv(CKPT_DIR / "grid_results.csv")
best = (grid.sort_values(["dev_top3", "dev_top1", "dev_loss"], ascending=[False, False, True]).groupby("model").head(1).set_index("model"))
sel = ROOT / "results" / "doctor" / "dev_selection_201.csv"
if sel.exists():                                                         # notebook 09's full-dev choice per size
    s = pd.read_csv(sel).sort_values(["dev201_top3", "dev201_top1", "dev_loss"], ascending=[False, False, True])
    best = s.groupby("model").head(1).set_index("model")
DOCTORS = {
    "NB doctor": lambda: NBDoctor(nb, lexicon, max_questions=MAX_QUESTIONS, seed=SEED),
    "GPT-2 small doctor": lambda: GPT2Doctor.from_pretrained(str(ROOT / best.loc["gpt2", "path"]), CONDITIONS),
    "GPT-2 medium doctor": lambda: GPT2Doctor.from_pretrained(str(ROOT / best.loc["gpt2-medium", "path"]), CONDITIONS),
}
available = {k: v for k, v in PATIENT_RUNS.items() if (PATIENT_CKPT / v / "config.json").exists()}
print("doctors:", list(DOCTORS), "| GPT-2 patients available:", list(available) or "none")

doctors: ['NB doctor', 'GPT-2 small doctor', 'GPT-2 medium doctor'] | GPT-2 patients available: ['GPT-2 small patient', 'GPT-2 medium patient']


## 2. Patients
`PatientSimulator` is the app's loader for the patient model (`simclinic/patient.py`). The adapter gives it the `reply(history)` interface that `run_consultation` expects. Notebook 08's sampling is kept (temperature 0.8, top-p 0.9).

In [3]:
class GPT2Patient:
    def __init__(self, sim, card):
        self.sim, self.card = sim, card

    def reply(self, history):
        # PatientSimulator puts "What brings you in today?" first when the history does not start with a doctor turn;
        # here it always does, so the history is passed as is.
        return self.sim.reply(self.card, history, temperature=0.8, top_p=0.9, max_new_tokens=80)   # notebook 08's GEN

def slug(s):
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")

def run(doctor_name, make_doctor, patient_name, make_patient):
    path = OUT_DIR / f"{slug(doctor_name)}__{slug(patient_name)}.jsonl"
    done = {r["card_id"] for r in read_jsonl(path)}
    todo = [c for c in test_cards if c["card_id"] not in done]
    if todo:
        t0, doctor = time.time(), make_doctor()
        with open(path, "a", encoding="utf-8") as f:
            for card in todo:
                set_seed(seed_of[card["card_id"]])
                rec = run_consultation(doctor, make_patient(card), max_questions=MAX_QUESTIONS)
                f.write(json.dumps({"doctor": doctor_name, "patient": patient_name, "card_id": card["card_id"],
                                    "disease": card["disease"], **score(rec, card, nb, lexicon), **rec},
                                   ensure_ascii=False) + "\n")
        del doctor
        gc.collect()
        torch.cuda.empty_cache()
        print(f"{doctor_name} x {patient_name}: {len(todo)} cases in {(time.time() - t0) / 60:.1f} min")
    return read_jsonl(path)

## 3. Run
The rule-based patient runs are notebook 05's setting; they are repeated here (same seeds) so every row of the result table comes from this notebook's code.

In [4]:
rows = []
for d_name, make_d in DOCTORS.items():
    rows += run(d_name, make_d, "rule-based patient", lambda card: CardPatient(card, lexicon, seed=seed_of[card["card_id"]]))
if available:
    from simclinic.patient import PatientSimulator
    for p_name, run_name in available.items():
        sim = PatientSimulator(str(PATIENT_CKPT / run_name))
        for d_name, make_d in DOCTORS.items():
            rows += run(d_name, make_d, p_name, lambda card, sim=sim: GPT2Patient(sim, card))
        del sim
        gc.collect()
        torch.cuda.empty_cache()
df = pd.DataFrame(rows)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

NB doctor x GPT-2 medium patient: 201 cases in 3.0 min


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 small doctor x GPT-2 medium patient: 201 cases in 5.1 min


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT-2 medium doctor x GPT-2 medium patient: 201 cases in 7.5 min


## 4. Results
### 4.1 Doctor accuracy by patient

In [5]:
doc = (df.groupby(["patient", "doctor"], sort=False)
         .agg(cases=("top1", "size"), top1=("top1", "mean"), top3=("top3", "mean"), nb_top1=("nb_top1", "mean"),
              questions=("n_questions", "mean"), card_symptoms_found=("symptom_recall", "mean"))
         .round(3))
doc.to_csv(RESULTS / "auto_doctor_by_patient.csv")
doc

cases   top1   top3  nb_top1  \
patient              doctor                                              
rule-based patient   NB doctor              201  0.706  0.940    0.706   
                     GPT-2 small doctor     201  0.687  0.861    0.542   
                     GPT-2 medium doctor    201  0.726  0.900    0.592   
GPT-2 small patient  NB doctor              201  0.567  0.846    0.567   
                     GPT-2 small doctor     201  0.766  0.960    0.517   
                     GPT-2 medium doctor    201  0.791  0.950    0.532   
GPT-2 medium patient NB doctor              201  0.567  0.861    0.567   
                     GPT-2 small doctor     201  0.831  0.965    0.552   
                     GPT-2 medium doctor    201  0.791  0.955    0.537   

                                          questions  card_symptoms_found  
patient              doctor                                               
rule-based patient   NB doctor                7.463                0.715  
                     GPT-2 small doctor       6.174                0.580  
                     GPT-2 medium doctor      6.353                0.616  
GPT-2 small patient  NB doctor                7.184                0.635  
                     GPT-2 small doctor       6.383                0.600  
                     GPT-2 medium doctor      6.706                0.600  
GPT-2 medium patient NB doctor                7.547                0.647  
                     GPT-2 small doctor       6.532                0.637  
                     GPT-2 medium doctor      6.682                0.643

### 4.2 Patient behaviour under each doctor
The same fidelity, coverage and leakage as notebook 08, but under free-form doctor questions.

In [6]:
def leak_names(d):
    names = [d, *re.findall(r"\(([^)]+)\)", d), re.sub(r"\s*\([^)]*\)", "", d), *C.EXTRA_SYNONYMS.get(d, [])]
    return [x for x in dict.fromkeys(names) if [(a, b) for a, b, _ in lexicon.find(x)] != [(0, len(x))]]

card_by_id = {c["card_id"]: c for c in test_cards}
pat = []
for r in rows:
    card = card_by_id[r["card_id"]]
    rev = revealed_symptoms(r["turns"], lexicon)
    has = [s for s, v in rev.items() if v]
    on = [s for s in has if lexicon.on_card(s, card["symptoms"])]
    replies = " ".join(t["text"] for t in r["turns"] if t["role"] == "patient")
    pat.append({"patient": r["patient"], "doctor": r["doctor"], "mentioned": len(has), "on_card": len(on),
                "leaked": bool(C.named_diseases(replies, leak_names(card["disease"])))})
pat = pd.DataFrame(pat)
pt = pat.groupby(["patient", "doctor"], sort=False).agg(mentioned=("mentioned", "sum"), on_card=("on_card", "sum"), leakage=("leaked", "mean"))
pt.insert(2, "symptom_fidelity", pt["on_card"] / pt["mentioned"])
pt.round(3).to_csv(RESULTS / "auto_patient_by_doctor.csv")
pt.round(3)

mentioned  on_card  \
patient              doctor                                    
rule-based patient   NB doctor                  603      603   
                     GPT-2 small doctor         484      484   
                     GPT-2 medium doctor        513      513   
GPT-2 small patient  NB doctor                  665      570   
                     GPT-2 small doctor         580      538   
                     GPT-2 medium doctor        587      546   
GPT-2 medium patient NB doctor                  648      579   
                     GPT-2 small doctor         577      558   
                     GPT-2 medium doctor        593      569   

                                          symptom_fidelity  leakage  
patient              doctor                                          
rule-based patient   NB doctor                       1.000    0.000  
                     GPT-2 small doctor              1.000    0.000  
                     GPT-2 medium doctor             1.000    0.000  
GPT-2 small patient  NB doctor                       0.857    0.000  
                     GPT-2 small doctor              0.928    0.005  
                     GPT-2 medium doctor             0.930    0.005  
GPT-2 medium patient NB doctor                       0.894    0.000  
                     GPT-2 small doctor              0.967    0.005  
                     GPT-2 medium doctor             0.960    0.000

### 4.3 Example consultation

In [7]:
if available:
    r = df[(df["doctor"] == "GPT-2 medium doctor") & (df["patient"] == list(available)[0])].iloc[3]
    card = card_by_id[r["card_id"]]
    print(f"hidden: {card['disease']} | card: {', '.join(card['symptoms'])}")
    for t in r["turns"]:
        print(f"  {t['role']:>7}: {t['text']}")
    print("top-3:", "; ".join(r["diagnosis"]))

hidden: Common cold | card: Cough, Fever, Coryza, Ear pain, Wheezing
   doctor: What brings you in today?
  patient: I've been coughing for the last two days.
   doctor: When did the cough start?
  patient: It started two days ago.
   doctor: Have you noticed any pain in your ears?
  patient: Yeah, my ears hurt a bit too.
   doctor: Do you have any other symptoms, like a headache?
  patient: No, I don't have a headache.
   doctor: Have you noticed any fluid coming out of your ear?
  patient: No, nothing like that.
   doctor: Do you have a fever or feel hot?
  patient: Yeah, I feel hot.
   doctor: Thank you for letting me know about your symptoms.
top-3: Common cold; Acute otitis media; Acute sinusitis
